In [1]:
import sqlite3

print("SQLite version:", sqlite3.sqlite_version)

SQLite version: 3.49.1


In [2]:
import pandas as pd
import sqlite3

# Connect to a local database file
conn = sqlite3.connect("online_retail.db")

# Load your cleaned CSV file
sales_df = pd.read_csv("online_retail_sales_clean.csv")

# Save the data as a SQL table
sales_df.to_sql(
    "retail_sales",
    conn,
    if_exists="replace",
    index=False
)

print("Data loaded successfully!")
print("Rows:", len(sales_df))
print("Columns:", sales_df.columns.tolist())

Data loaded successfully!
Rows: 504176
Columns: ['Invoice', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'Price', 'Customer ID', 'Country', 'Revenue', 'YearMonth']


In [3]:
query = """
SELECT SUM(Revenue) AS total_revenue
FROM retail_sales;
"""

result = pd.read_sql_query(query, conn)
print(result)

   total_revenue
0   1.000917e+07


In [4]:
query = """
SELECT
    Description,
    ROUND(SUM(Revenue), 2) AS total_revenue
FROM retail_sales
GROUP BY Description
ORDER BY total_revenue DESC
LIMIT 5;
"""

result = pd.read_sql_query(query, conn)
print(result.to_string(index=False))

                       Description  total_revenue
          REGENCY CAKESTAND 3 TIER      169912.76
WHITE HANGING HEART T-LIGHT HOLDER      160345.63
                    DOTCOM POSTAGE      116408.71
     ASSORTED COLOUR BIRD ORNAMENT       72890.19
   PAPER CHAIN KIT 50'S CHRISTMAS        58127.30


In [5]:
query = """
SELECT
    Country,
    ROUND(SUM(Revenue), 2) AS total_revenue
FROM retail_sales
GROUP BY Country
ORDER BY total_revenue DESC
LIMIT 5;
"""

result = pd.read_sql_query(query, conn)
print(result.to_string(index=False))

       Country  total_revenue
United Kingdom     8601663.65
          EIRE      364099.45
   Netherlands      268587.67
       Germany      201250.37
        France      142082.44


In [6]:
query = """
SELECT
    SUBSTR(InvoiceDate, 1, 7) AS sales_month,
    ROUND(SUM(Revenue), 2) AS monthly_revenue
FROM retail_sales
GROUP BY sales_month
ORDER BY sales_month;
"""

result = pd.read_sql_query(query, conn)
print(result.to_string(index=False))

sales_month  monthly_revenue
    2009-12        820246.56
    2010-01        635533.94
    2010-02        547305.20
    2010-03        776030.64
    2010-04        655504.64
    2010-05        652620.04
    2010-06        706357.74
    2010-07        643390.21
    2010-08        684520.00
    2010-09        880612.74
    2010-10       1109163.84
    2010-11       1461357.62
    2010-12        436529.63


In [7]:
query = """
SELECT
    "Customer ID",
    ROUND(SUM(Revenue), 2) AS total_revenue,
    COUNT(DISTINCT Invoice) AS number_of_orders
FROM retail_sales
WHERE "Customer ID" IS NOT NULL
GROUP BY "Customer ID"
ORDER BY total_revenue DESC
LIMIT 5;
"""

result = pd.read_sql_query(query, conn)
print(result.to_string(index=False))

 Customer ID  total_revenue  number_of_orders
     18102.0      349164.35                89
     14646.0      248396.50                78
     14156.0      188371.29                97
     14911.0      144115.90               188
     13694.0      131443.19                94


In [8]:

query = """
SELECT
    COUNT(*) AS total_customers,
    SUM(CASE WHEN number_of_orders > 1 THEN 1 ELSE 0 END) AS repeat_customers,
    ROUND(
        100.0 * SUM(CASE WHEN number_of_orders > 1 THEN 1 ELSE 0 END) / COUNT(*),
        2
    ) AS repeat_customer_rate
FROM (
    SELECT
        "Customer ID",
        COUNT(DISTINCT Invoice) AS number_of_orders
    FROM retail_sales
    WHERE "Customer ID" IS NOT NULL
    GROUP BY "Customer ID"
);
"""

result = pd.read_sql_query(query, conn)
print(result.to_string(index=False))


 total_customers  repeat_customers  repeat_customer_rate
            4298              2880                 67.01
